## Image classification with CNN

Dans ce fichier je vais récupérer une basse de donnée comprenant des images de nombre avec un label qui eplisite ce nombre. C'est un exercise de base que je fait en paralléle d'une vidéo de 3Blue1Brown sur le Deep Learning.

In [2]:
import argparse
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

In [3]:
train_dataset = datasets.MNIST(root='./data', train=True, download=True)
# root : zone de stokage des données, train = True : les donnée charger sont celle d'entrainement

In [4]:
# Creation d'un CNN 

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)         # Couches de convolution, pour la 1 : entrer de taille 1 image MNIST, produit 32 carte de caractérisation, utilise un noyau 3x3 et un stride de 1
        self.conv2 = nn.Conv2d(32, 64, 3, 1)        # Apres la couche de convolution 1 la taille des image passe de 28 à 26 et aprés la 2 de 26 à 24
        self.dropout1 = nn.Dropout(0.25)            # Couches de régularisation : met a 0 25% puis 50% des neurones de maniere a éviter le surapprentissage
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)             # Couches entiérement connectée : prend une entrée de taille 9216 vers une taille 128
        self.fc2 = nn.Linear(128, 10)               # Si on a besoins de 9216 neurones c'est car aprés les deux couche de convolution on a 64 filtre et une image de taill 24x24 qui a était réduit à 12x12 avec un pooling : 12x12x64 = 9216

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)                               # introduit la non linérité
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)                      # Couche pooling réduit la dimension de l'image obtenue
        x = self.dropout1(x)                        
        x = torch.flatten(x, 1)                     # transforme un vecteur transforme le tenseur 4D (batch, canaux, hauteur, largeur) en un vecteur 2D (batch, features). (On est en 2D car les opération sont effectuer en paralléle)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

In [5]:
# args : paramétres
# model : NN
# device : cpu ou gpu
# train_loader : charge les batch de données
# optimizer : algorithme d'optimisation ça peut étre SGD ou Adam ou autre
# epoch : numéro de l'époque actuelle


def train(args, model, device, train_loader, optimizer, epoch):
    model.train()   # Passe le modéle en mode train (Important car en donction du mode certain couche ce conporte différament, les couche de dropout par example)
    for batch_idx, (data, target) in enumerate(train_loader):   # On parcous les données, batcj_idx est l'ID, data l'image est target le label.
        data, target = data.to(device), target.to(device)    # On envoie les données sur le bonne apprareil (C'est géneralement pour le passer sur le GPU pour accélerer les calculs)
        optimizer.zero_grad()   # On remet le gradiant à zero (Par défault pytoch accumule les gradiants)
        output = model(data)    # On fait passer une image dans le modéle
        loss = F.nll_loss(output, target)   # Calcul de la loss entre la cible et l'output
        loss.backward()     # calcule les gradients (la rétropropagation mathématique) 
        optimizer.step()    # applique ces gradients pour modifier les poids
        if batch_idx % args.log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(epoch, batch_idx * len(data), len(train_loader.dataset), 100. * batch_idx / len(train_loader), loss.item()))
            if args.dry_run:
                break

In [6]:
def test(model, device, test_loader):
    model.eval()    # passage du modéle en mode éval désactivation de certaines couche
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    # Le .item() permet d'extraire la valeur numérique d'un tenseur qui ne contient qu'un unique tenseur pour la convertir en un nombre python classique (float ou int)

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

In [7]:
def main():
    # Le parser d'arguments(argparse) permet de passer des options au programme via la ligne de commande
    # Example : python mnist.py --batch-size 128 --epochs 10
    parser = argparse.ArgumentParser(description='PyTorch MNIST Example')
    parser.add_argument('--batch-size', type=int, default=64, metavar='N', help='input batch size for training (default: 64)')
    parser.add_argument('--test-batch-size', type=int, default=1000, metavar='N', help='input batch size for testing (default: 1000)')
    parser.add_argument('--epochs', type=int, default=14, metavar='N', help='number of epochs to train (default: 14)')
    parser.add_argument('--lr', type=float, default=1.0, metavar='LR', help='learning rate (default: 1.0)')
    parser.add_argument('--gamma', type=float, default=0.7, metavar='M', help='Learning rate step gamma (default: 0.7)')
    parser.add_argument('--no-accel', action='store_true', help='disables accelerator')
    parser.add_argument('--dry-run', action='store_true', help='quickly check a single pass')
    parser.add_argument('--seed', type=int, default=1, metavar='S', help='random seed (default: 1)')
    parser.add_argument('--log-interval', type=int, default=10, metavar='N', help='how many batches to wait before logging training status')
    parser.add_argument('--save-model', action='store_true', help='For Saving the current Model')
    # args = parser.parse_args()
    args, unknown = parser.parse_known_args()

    # Choix de l'appareil (torch.accelerator / CPU) Initialement dans le code on utiliser torch.accelerator pour effectuer les calcul qui a un avantage certain sur cuda c'est qu'il tourne sur plus de support GPU AMD/NVIDIA, TPU, MEtal,... alors que cuda ne tourne que des GPU NVIDIA
    # use_accel = not args.no_accel and torch.accelerator.is_available()
    # torch.manual_seed(args.seed)

    # if use_accel:
    #     device = torch.accelerator.current_accelerator()
    # else:
    #     device = torch.device("cpu")

    torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")



    # Preparation des datas
    train_kwargs = {'batch_size': args.batch_size}
    test_kwargs = {'batch_size': args.test_batch_size}
    
    if device.type == 'cuda':
        accel_kwargs = {'num_workers': 1, 'persistent_workers': True, 'pin_memory': True, 'shuffle': True}
        train_kwargs.update(accel_kwargs)
        test_kwargs.update(accel_kwargs)

    # Transformation des données : conversion en tensor pytorch normalizer
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])  
    dataset1 = datasets.MNIST('../data', train=True, download=True, transform=transform)
    dataset2 = datasets.MNIST('../data', train=False, transform=transform)
    train_loader = torch.utils.data.DataLoader(dataset1,**train_kwargs)
    test_loader = torch.utils.data.DataLoader(dataset2, **test_kwargs)

    model = Net().to(device)    # Chargement de notre CNN
    optimizer = optim.Adadelta(model.parameters(), lr=args.lr)  # Utilisation de l'optimizer Adadelta : ajuste automatiqueent les poids selon le gradient avec un learning rate initial args.lr

    scheduler = StepLR(optimizer, step_size=1, gamma=args.gamma) # Diminue le learning rate tous les step_size epochs.
    for epoch in range(1, args.epochs + 1):
        train(args, model, device, train_loader, optimizer, epoch)
        test(model, device, test_loader)
        scheduler.step()

    if args.save_model:
        torch.save(model.state_dict(), "mnist_cnn.pt")

if __name__ == '__main__':
    main()

Using device: cuda
Train Epoch: 1 [0/60000 (0%)]	Loss: 2.282942
Train Epoch: 1 [640/60000 (1%)]	Loss: 1.289376
Train Epoch: 1 [1280/60000 (2%)]	Loss: 0.955769
Train Epoch: 1 [1920/60000 (3%)]	Loss: 0.690552
Train Epoch: 1 [2560/60000 (4%)]	Loss: 0.414190
Train Epoch: 1 [3200/60000 (5%)]	Loss: 0.438484
Train Epoch: 1 [3840/60000 (6%)]	Loss: 0.272626
Train Epoch: 1 [4480/60000 (7%)]	Loss: 0.445310
Train Epoch: 1 [5120/60000 (9%)]	Loss: 0.256343
Train Epoch: 1 [5760/60000 (10%)]	Loss: 0.133453
Train Epoch: 1 [6400/60000 (11%)]	Loss: 0.213857
Train Epoch: 1 [7040/60000 (12%)]	Loss: 0.257410
Train Epoch: 1 [7680/60000 (13%)]	Loss: 0.303448
Train Epoch: 1 [8320/60000 (14%)]	Loss: 0.128087
Train Epoch: 1 [8960/60000 (15%)]	Loss: 0.457085
Train Epoch: 1 [9600/60000 (16%)]	Loss: 0.057561
Train Epoch: 1 [10240/60000 (17%)]	Loss: 0.295271
Train Epoch: 1 [10880/60000 (18%)]	Loss: 0.249255
Train Epoch: 1 [11520/60000 (19%)]	Loss: 0.216213
Train Epoch: 1 [12160/60000 (20%)]	Loss: 0.141861
Train Epoc

False
